# Matrix Alpha // Firestarter SPB
## Firestarter-Style Top 100 Multi-Panel Chart Viewer

> **[IMPORTANT] WARNINGS & CONSTRAINTS**
> - **Research Viewer Only:** This notebook is strictly for offline inspection, profiling, and quality validation of the ingested dataset.
> - **No Cell 2 / Labels:** This notebook does not generate target labels, features, or predictive variables.
> - **No Model Training:** There is no machine learning model training or parameter optimization here.
> - **No Trading Recommendations / Strategy Claims:** This notebook does not contain any trading strategies, execution logic, or financial recommendations.
> - **No ER / FMLC / Flowprint Computation:** No custom volatility or flow-based indicator calculation is performed in this version.

In [ ]:
import os
import glob
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone

# Set styling for premium visual feel
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (14, 16)
plt.rcParams['font.size'] = 10
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8

# Dataset Path
DATA_DIR = "../data/research/binance_top100_excluding_existing_5_1month"
print(f"Dataset target path configured: {DATA_DIR}")

### Symbol and Date Range Selection

**Instruction:** 
To analyze a different symbol or time slice:
1. Update the `SELECTED_SYMBOL` variable to any valid ticker.
2. Customize `START_DATE` and `END_DATE` (format: `'YYYY-MM-DD'`) or leave as `None` to inspect the full range.
3. Run the cells below.

In [ ]:
# UPDATE THESE VARIABLES FOR CUSTOM SLICING
SELECTED_SYMBOL = "BTCUSDT"
START_DATE = None  # e.g., "2026-06-01"
END_DATE = None    # e.g., "2026-06-07"

print(f"Selected Ticker: {SELECTED_SYMBOL}")
print(f"Custom Window: Start={START_DATE}, End={END_DATE}")

### 1. Ingested Dataset Inventory Audit
We scan the target directory to verify the data integrity of all 100 symbols.

In [ ]:
def build_inventory(data_dir):
    csv_files = glob.glob(os.path.join(data_dir, "*_1month_5m.csv"))
    inventory = []
    
    for filepath in csv_files:
        filename = os.path.basename(filepath)
        symbol = filename.replace("_1month_5m.csv", "")
        
        try:
            df_temp = pd.read_csv(filepath)
            row_count = len(df_temp)
            
            if row_count > 0:
                first_ts = int(df_temp.iloc[0]['open_time'])
                last_ts = int(df_temp.iloc[-1]['open_time'])
                
                expected_span_count = int((last_ts - first_ts) / 300000) + 1
                missing_5m = max(0, expected_span_count - row_count)
                
                first_dt = pd.to_datetime(first_ts, unit='ms', utc=True).strftime('%Y-%m-%d %H:%M:%S')
                last_dt = pd.to_datetime(last_ts, unit='ms', utc=True).strftime('%Y-%m-%d %H:%M:%S')
            else:
                first_ts, last_ts = np.nan, np.nan
                first_dt, last_dt = "N/A", "N/A"
                missing_5m = 0
                
            nonstandard_flag = not symbol.isascii()
            
            inventory.append({
                "symbol": symbol,
                "row_count": row_count,
                "first_timestamp": first_ts,
                "last_timestamp": last_ts,
                "first_time_utc": first_dt,
                "last_time_utc": last_dt,
                "missing_5m_count": missing_5m,
                "nonstandard_symbol_flag": nonstandard_flag
            })
        except Exception as e:
            print(f"Error reading {symbol}: {e}")
            
    df_inv = pd.DataFrame(inventory)
    if not df_inv.empty:
        df_inv = df_inv.sort_values(by="symbol")
    return df_inv

df_inventory = build_inventory(DATA_DIR)
print(f"Loaded metadata for {len(df_inventory)} symbols.")

# Check for non-standard symbols and trigger warning flag
nonstandard_symbols = df_inventory[df_inventory['nonstandard_symbol_flag'] == True]['symbol'].tolist()
if nonstandard_symbols:
    print(f"\n[WARNING] Non-standard Unicode tickers detected in dataset: {nonstandard_symbols}")

df_inventory

### 2. Selected Symbol Detailed Profiling & Data Loading

In [ ]:
csv_path = os.path.join(DATA_DIR, f"{SELECTED_SYMBOL}_1month_5m.csv")
if not os.path.exists(csv_path):
    files = glob.glob(os.path.join(DATA_DIR, f"*{SELECTED_SYMBOL}*_5m.csv"))
    if files:
        csv_path = files[0]
        print(f"Resolved non-standard path: {csv_path}")
    else:
        raise FileNotFoundError(f"No CSV found for: {SELECTED_SYMBOL}")

df = pd.read_csv(csv_path)
df['open_datetime'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
df['close_datetime'] = pd.to_datetime(df['close_time'], unit='ms', utc=True)

# Slicing time window if configured
if START_DATE:
    df = df[df['open_datetime'] >= pd.to_datetime(START_DATE, utc=True)]
if END_DATE:
    df = df[df['open_datetime'] <= pd.to_datetime(END_DATE, utc=True)]

print(f"Successfully loaded {len(df)} rows for {SELECTED_SYMBOL}")
if not SELECTED_SYMBOL.isascii():
    print("[WARNING] This symbol contains nonstandard unicode characters. Handle encoding safely.")

print("\n--- Latest 20 Rows Preview ---")
df.tail(20)

### 3. Numeric Summary Statistics & Volatility Slicing

In [ ]:
print("--- OHLCV Numeric Stats ---")
print(df[['open', 'high', 'low', 'close', 'volume']].describe())

df['candle_range'] = df['high'] - df['low']
df['range_pct'] = (df['candle_range'] / df['open']) * 100

print("\n--- Volatility and Volume Analysis ---")
vol_stats = {
    "Total Volume (Base)": df['volume'].sum(),
    "Average Volume/Candle": df['volume'].mean(),
    "Max Volume/Candle": df['volume'].max(),
    "Mean Candle Range (USDT)": df['candle_range'].mean(),
    "Mean Candle Range (% of Open)": df['range_pct'].mean(),
    "Max Candle Range (% of Open)": df['range_pct'].max(),
}
for k, v in vol_stats.items():
    print(f"{k}: {v:,.4f}")

### 4. Multi-Scale Timeframe Aggregation (1h and 4h)
We resample the 5-minute ticks to 1-hour and 4-hour aggregates and construct summary tables.

In [ ]:
df_resample = df.set_index('open_datetime')

# 1-Hour Aggregation
df_1h = df_resample.resample('1h').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum',
    'quote_asset_volume': 'sum',
    'trades': 'sum'
}).dropna()

# 4-Hour Aggregation
df_4h = df_resample.resample('4h').agg({
    'open': 'first',
    'high': 'max',
    'low': 'min',
    'close': 'last',
    'volume': 'sum',
    'quote_asset_volume': 'sum',
    'trades': 'sum'
}).dropna()

print("\n--- 1-Hour Aggregated Summary Stats ---")
print(df_1h.describe())

print("\n--- 4-Hour Aggregated Summary Stats ---")
print(df_4h.describe())

### 5. Firestarter-Style Multi-Panel Visualization
We construct a 4-panel visual replay sheet showing Close Price with trend lines, color-coded volume, high-low range %, and rolling volatility.

In [ ]:
# Compute price trend lines (SMAs)
df_1h['sma_20'] = df_1h['close'].rolling(20).mean()
df_1h['sma_50'] = df_1h['close'].rolling(50).mean()

# Volume colors: Green if close >= open (bullish), Red otherwise (bearish)
df_1h['vol_color'] = np.where(df_1h['close'] >= df_1h['open'], '#26a69a', '#ef5350')

# High-Low Candle spread %
df_1h['range_pct'] = ((df_1h['high'] - df_1h['low']) / df_1h['open']) * 100

# 20-period rolling volatility (standard deviation of log returns)
df_1h['log_ret'] = np.log(df_1h['close'] / df_1h['close'].shift(1))
df_1h['rolling_vol'] = df_1h['log_ret'].rolling(20).std() * 100

# Setup plot axes
fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True, gridspec_kw={'height_ratios': [2, 1, 1, 1]})

# Panel 1: Price and Trend
axes[0].plot(df_1h.index, df_1h['close'], color='#2196f3', linewidth=1.5, label='1h Close Price')
axes[0].plot(df_1h.index, df_1h['sma_20'], color='#ff9800', linewidth=1.0, linestyle='--', label='20 SMA')
axes[0].plot(df_1h.index, df_1h['sma_50'], color='#e91e63', linewidth=1.0, linestyle='--', label='50 SMA')
axes[0].set_ylabel('Price (USDT)', fontweight='bold')
axes[0].set_title(f'{SELECTED_SYMBOL} - Firestarter Profile Viewer', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper left')
axes[0].grid(True, alpha=0.3)

# Panel 2: Color-Coded Volume
axes[1].bar(df_1h.index, df_1h['volume'], color=df_1h['vol_color'], width=0.03, alpha=0.8, label='Volume')
axes[1].set_ylabel('Volume', fontweight='bold')
axes[1].legend(loc='upper left')
axes[1].grid(True, alpha=0.3)

# Panel 3: High-Low Range %
axes[2].plot(df_1h.index, df_1h['range_pct'], color='#9c27b0', linewidth=1.2, label='High-Low Range %')
axes[2].set_ylabel('Range %', fontweight='bold')
axes[2].legend(loc='upper left')
axes[2].grid(True, alpha=0.3)

# Panel 4: Rolling Volatility %
axes[3].plot(df_1h.index, df_1h['rolling_vol'], color='#00bcd4', linewidth=1.2, label='20-Period Rolling Vol %')
axes[3].set_ylabel('Volatility %', fontweight='bold')
axes[3].set_xlabel('Time (UTC)', fontweight='bold')
axes[3].legend(loc='upper left')
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()